In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)
from pyspark.sql.functions import lit, current_timestamp
import re

# ============================================================
# PARÂMETROS
# ============================================================

CAMINHO_CSV = "/Volumes/mba/stage/dados_bruto/ONS/ONS-ENA_DIARIO_RESERVATORIOS_2001.csv"

CAMINHO_ORIGEM = "/Volumes/mba/stage/dados_bruto/ONS"

TABELA_DESTINO = "mba.raw.ons_ena_diario_reservatorios"


# ============================================================
# SCHEMA DO ARQUIVO CSV
# ============================================================

schema_ena = StructType([    
    StructField("nom_reservatorio", StringType(), True),    
    StructField("cod_resplanejamento", IntegerType(), True),    
    StructField("tip_reservatorio", StringType(), True),    
    StructField("nom_bacia", StringType(), True),    
    StructField("nom_ree", StringType(), True),    
    StructField("id_subsistema", StringType(), True),    
    StructField("nom_subsistema", StringType(), True),    
    StructField("ena_data", DateType(), True),    
    StructField("ena_bruta_res_mwmed", DoubleType(), True),    
    StructField("ena_bruta_res_percentualmlt", DoubleType(), True),    
    StructField("ena_armazenavel_res_mwmed", DoubleType(), True),    
    StructField("ena_armazenavel_res_percentualmlt", DoubleType(), True),    
    StructField("ena_queda_bruta", DoubleType(), True),    
    StructField("mlt_ena", DoubleType(), True)
])

In [0]:
# ============================================
# LISTAGEM DOS ARQUIVOS
# ============================================

CAMINHO_ORIGEM = "/Volumes/mba/stage/dados_bruto/ONS/"
arquivos = dbutils.fs.ls(CAMINHO_ORIGEM)

arquivos = [
    arquivo
    for arquivo in arquivos
    if re.match(
        r"^ONS-ENA_DIARIO_RESERVATORIOS_\d{4}\.csv$",
        arquivo.name,
        re.IGNORECASE
    )
]

if len(arquivos) == 0:
    dbutils.notebook.exit("Nenhum arquivo ONS encontrado.")

print(f"Arquivos ONS encontrados: {len(arquivos)}")

In [0]:
# Limpa tabela
spark.sql(f"truncate table {TABELA_DESTINO}")

# LEITURA DO CSV
for arquivo in arquivos:
    print(f"Processando arquivo: {arquivo.name}")
    caminho_arquivo = arquivo.path.replace("dbfs:", "")

    # EXTRAI O NOME DO ARQUIVO
    arq = caminho_arquivo.split("/")[-1]

    df_ena = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("dateFormat", "yyyy-MM-dd")
        .option("nullValue", "")
        .schema(schema_ena)
        .load(caminho_arquivo)
    )

    # ADICIONA COLUNAS DE CONTROLE

    df_ena = (
        df_ena
        .withColumn("NomeArquivo", lit(arq))
        .withColumn("DataCarga", current_timestamp())
    )

    # CARGA DA DELTA TABLE
    df_ena.write \
        .mode("append") \
        .saveAsTable(TABELA_DESTINO) 
